## 📓 About this notebook
This notebook sets up production monitoring for the deployed Unity Airways assistant: registering quality scorers, enabling online monitoring on live traces, simulating traffic, capturing human feedback, and mining traces to grow the evaluation dataset.

**Maps to the book:** Chapter 9, *Production Monitoring with MLflow* — sections: Types of Metrics to Monitor, Online Monitoring Workflow, Advanced Monitoring on Production Traces, Expanding the Evaluation Dataset.

**Compute:** Serverless Standard v5

In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import yaml

with open('../conf/data.yml') as f:
    default_uc = yaml.safe_load(f)['default_uc']

experiment_path = '/Shared/deployed_agent'
catalog = default_uc['catalog']
schema = default_uc['schema']
eval_data_table_name = 'eval_data'

## Register Scorers
Here, we register scorers for online monitoring. In practice, these scorers should be consistent with those used during the development phase to ensure that evaluation metrics remain comparable across development and production.

In [0]:
import mlflow 
from mlflow.genai.scorers import (
    RelevanceToQuery,
    Safety,
    RetrievalGroundedness,
)

mlflow.set_experiment(experiment_path)

In [0]:
eval_model = "databricks:/databricks-gpt-oss-120b"
ua_relevancy_scorer = RelevanceToQuery(model=eval_model)
ua_safety_scorer = Safety(model=eval_model)
ua_groundedness_scorer = RetrievalGroundedness(model=eval_model)

scorer_list = [
    ua_relevancy_scorer,
    ua_safety_scorer,
    ua_groundedness_scorer,
]

In [0]:
from mlflow.genai.scorers import list_scorers

existing_scorers = [i.name for i in list_scorers()]

for scorer in scorer_list:
    if scorer.name in existing_scorers:
      print(f"Scorer already exists: {scorer}")
    else:
      registered_scorer = scorer.register(name=scorer.name)
      print(f"Registered scorer: {scorer}")

### Create Evaluation Dataset
We simulate an evaluation dataset that will later be expanded with simulated production requests.

In [0]:
eval_data_df = spark.read.table(
    f"{catalog}.{schema}.qa_dataset"
).limit(50)

# Create an evaluation 

try:
  eval_dataset_new = mlflow.genai.datasets.create_dataset(
      name=f"{catalog}.{schema}.eval_dataset_chang_temp",
  )
  print(f"Created evaluation dataset: {catalog}.{schema}.eval_dataset_chang_temp")
except Exception as e:
  if e.response.json()['error_code'] == 'TABLE_ALREADY_EXISTS':
    print('Evaluation dataset is already created')
    eval_dataset_new = mlflow.genai.datasets.get_dataset(f"{catalog}.{schema}.eval_dataset_chang_temp")
  else:
    print('Unxpected error')


# Transform columns to match evaluation dataset schema
imported_data = eval_data_df.selectExpr(
    "struct(Question as question) as inputs",
    "struct(Answer as expected_response) as expectations"
)
# Update the dataset with imported cases
eval_dataset_new = eval_dataset_new.merge_records(imported_data)

### Enable Online Monitoring
The scorers registered in MLflow must be enabled for online monitoring.

In [0]:
from mlflow.genai.scorers import get_scorer

# Load scorers
ua_relevancy_scorer = get_scorer(name="relevance_to_query")
ua_safety_scorer = get_scorer(name="safety")
ua_groundedness_scorer = get_scorer(name="retrieval_groundedness")

# Assemble scorers into a list
unity_airways_scorers = [
    ua_relevancy_scorer,
    ua_safety_scorer,
    ua_groundedness_scorer
]

In [0]:
from mlflow.genai.scorers import ScorerSamplingConfig

for scorer in unity_airways_scorers:
    scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))

## Simulate Proudction Traffic
Now we send a few requests to the endpoint we deployed in chapter 7

In [0]:
sample_questions = [
    "Is airport check-in available for international Unity Airways flights?",
    "How early should I arrive at the airport for Unity Airways check-in?",
    "Can I check in multiple passengers under the same Unity Airways booking at once?",
    "What should I do if I can’t access my Unity Airways boarding pass after check-in?",
    "Will Unity Airways charge a fee if I cancel my flight before departure?",
    "How long does it take to receive a refund after canceling a Unity Airways ticket?",
    "Can I cancel only one passenger from a group booking with Unity Airways?",
    "Are musical instruments allowed as cabin baggage on Unity Airways?",
    "What happens if my Unity Airways baggage exceeds the allowed weight at the airport?",
    "Can I pay for additional baggage at the Unity Airways check-in counter?",
]

In [0]:

import os
from databricks.sdk import WorkspaceClient

# Get credentials and host URL from WorkspaceClient
ws = WorkspaceClient()
DATABRICKS_TOKEN = ws.tokens.create().token_value
DATABRICKS_HOST = ws.config.host

In [0]:
def predict_endpoint(question):

    import requests

    import os
    from databricks.sdk import WorkspaceClient

    # Get credentials and host URL from WorkspaceClient
    ws = WorkspaceClient()
    DATABRICKS_TOKEN = ws.tokens.create().token_value
    DATABRICKS_HOST = ws.config.host

    # Update name of endpoint if neccessary
    workspace_serving_url = f"{DATABRICKS_HOST}/serving-endpoints/responses"
    model_serving_name = "agents_workspace-unity_airways-unity-airways-booking-agent"

    headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
    }


    payload = {
    "model": model_serving_name,
    "input": [
        {"role": "user", "content": question}
    ]
    }

    response = requests.post(workspace_serving_url, headers=headers, json=payload, stream=True)

    return response

In [0]:
for question in sample_questions:
    predict_endpoint(question)

## Human Feedback
In production, we can also collect human feedback and log it to MLflow by associating each feedback entry with the corresponding `trace_id`.

In [0]:
# Unity Airways: capturing end-user feedback
import mlflow
from typing import Optional
from mlflow.entities.assessment import AssessmentSource, AssessmentSourceType
def log_end_user_feedback(
    trace_id: str,
    satisfied: bool,
    rationale: Optional[str] = None,
    user_id: Optional[str] = None,
) -> dict:
    """
    Record a thumbs-up or thumbs-down from the chat UI against a specific trace.
    - trace_id: the MLflow trace identifier returned with the chat response
    - satisfied: True for thumbs-up, False for thumbs-down
    - rationale: optional short comment or category label
    - user_id: optional application user identifier
    """
    mlflow.log_feedback(
        trace_id=trace_id,
        name="user_feedback",
        value=satisfied,
        rationale=rationale,
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id=user_id,
        ),
    )
    return {"status": "success", "trace_id": trace_id}

In [0]:
trace_id = "tr-43327f5eb8201db1ffaef99ee15540c1"
statisfied = False
rationale = "The answer should specify the fees charged when baggage limits are exceeded."
user_id = "max.lim@example.com"

log_end_user_feedback(trace_id, statisfied, rationale, user_id)

## Analyze Production Traces
Traces can be stored in Unity Catalog tables for further analysis. For example, we can use Databricks `ai_query` to analyze the questions asked by our users.

In [0]:
%sql
SELECT 
  -- 1. Get the input content
  get_json_object(request, '$.request.input[0].content') AS content_value,
  
  -- 2. Run AI Query and immediately extract the 'category' field from the JSON result
  get_json_object(
    ai_query(
      "databricks-gpt-oss-120b", 
      "Classify the intent of this question into one of these categories: Check-in, Cancellation, Baggage. Question: " || get_json_object(request, '$.request.input[0].content'),
      responseFormat => '{
        "type": "json_schema",
        "json_schema": {
          "name": "intent_classification",
          "schema": {
            "type": "object",
            "properties": {
              "category": { 
                "type": "string",
                "enum": ["Check-in", "Cancellation", "Baggage", "Others"]
              }
            },
            "required": ["category"],
            "additionalProperties": false
          },
          "strict": true
        }
      }'
    ),
    '$.category'
  ) AS intent_category
FROM 
  workspace.unity_airways.production_traces;

## Update Evaluation Dataset
We can now search for the latest production traces and use them to expand our evaluation dataset for the next iteration of the agent. In practice, we should be more selective about which traces are added to the evaluation dataset.

In [0]:
traces = mlflow.search_traces(
    filter_string=("traces.status = 'OK'"),
    order_by=["attributes.timestamp_ms DESC"],
    max_results=10
)
print(f"Found {len(traces)} successful traces")

if len(traces) > 0:
    # Transform traces to match evaluation dataset schema
    # Extract question from request and response from response field
    import pandas as pd
    
    transformed_data = pd.DataFrame({
        'inputs': [{'question': trace.get('request', {}).get('input', [{}])[0].get('content', '')} for _, trace in traces.iterrows()],
        'expectations': [{'expected_response': trace.get('response', '')} for _, trace in traces.iterrows()]
    })
    
    # Add the traces to the evaluation dataset
    eval_dataset_new = eval_dataset_new.merge_records(transformed_data)
    print(f"Added {len(traces)} records to evaluation dataset")
else:
    print("No traces to add - skipping dataset update")

In [0]:
import mlflow

mlflow.search_traces(
    filter_string="feedback.user_feedback = 'False'",
    max_results=10
)